# Test Synthethic Data Generation (SDG) AUG-PE Algorithm

The idea of this algorithm is to generate synthetic copies of sensitive text data using an LLM with an API without the LLM ever reading any of the real data. The algorithm also includes a Differential Privacy (DP) garuantee that ensure that the removal of any one record does not impact the distribution of the data, strenghthening the privacy protections of the algorithm. 

I think in a proper implementation I would set up two classes, one that included the public data and api access and one that had the private data in a hidden variable just to avoid an overlap.

# Step 0 - Input Variables

Generate some fake restuarant reviews for testing and some of the global AUG-PE parameters.

In [9]:
import anthropic
import os
import numpy as np
import pandas as pd
from great_tables import GT, style, loc
import pandas as pd
from sentence_transformers import SentenceTransformer
from huggingface_hub import snapshot_download

In [10]:
# Restaurant reviews
CATEGORIES = ["acting", "plot", "visuals"]

private_data = [
    # Single category
    "The lead actor delivered an absolutely mesmerizing performance that carried the entire film.",
    "The storyline was predictable and full of plot holes that left me frustrated.",
    "The cinematography was breathtaking, every frame looked like a painting.",
    "The CGI was cheap and unconvincing, completely pulling me out of the experience.",
    
    # Two categories
    "The plot was gripping and kept me on the edge of my seat, but the acting felt wooden and unconvincing.",
    "Stunning visuals throughout but the story made absolutely no sense by the third act.",
    "The cast was phenomenal and clearly committed to their roles, but the script let them down badly.",
    
    # Three categories
    "A visual masterpiece with a twisting plot and career-best performances from the entire cast.",
    "Terrible acting, a nonsensical storyline, and visuals that looked like they were made for TV in the 90s.",
    "The actors clearly tried their best but the weak plot and murky washed-out visuals made it a slog to sit through."
]

# Global Varibales

N_SYNTHETIC = 10

EPSILON = 100000#4.0        # less noise, paper shows this still gives good privacy
N_CANDIDATES = 1000    # more diverse initial pool, note technically this should be n_syn * number of variations.
T = 5                # more iterations to spread out
N_VARIATIONS = 6     # variations per resampled text per iteration


In [11]:
# Connect to anthropic API and load the encoding model
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# TODO This model should be sentiment aware - T
#encoder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
encoder = SentenceTransformer('./bge_finetuned_movie_reviews')

# Step 1 - DP_NN_HISTOGRAM

DP_NN_HISTOGRAM constructs a histogram of the nearest neighbours of the real data compared to the synthethic data. Distance is based on the embedding model $\Phi$. Then random gaussian noise $\frac{1}{\epsilon}$ is added to enforce DP privacy. This histogram is then used to draw samples from.

At the most basic level, it is just counting for each synthethic record how many real records is the synthetic record the most similar to. Then we draw from this distribution to get a distribution that should match the distribution of the real data.

In [12]:
# --- DP-NN HISTOGRAM ---
def dp_nn_histogram(candidates, private_texts, epsilon, n_resample):
    priv_embs = encoder.encode(private_texts, normalize_embeddings=True)
    cand_embs = encoder.encode(candidates, normalize_embeddings=True)

    counts = np.zeros(len(candidates))
    for emb in priv_embs:
        best_idx = int(np.argmax(cand_embs @ emb))
        counts[best_idx] += 1

    print(f"  Raw counts:   {counts.astype(int).tolist()}")
    noisy_counts = counts + np.random.normal(0, 1.0 / epsilon, size=counts.shape)
    noisy_counts = np.clip(noisy_counts, 0, None)
    probs = noisy_counts / noisy_counts.sum()
    print(f"  Resample probs: {[f'{p:.2f}' for p in probs]}")

    top_indices = np.argsort(probs)[::-1][:N_SYNTHETIC]
    resampled = [candidates[i] for i in top_indices]
    print(f"  Resampled pool ({len(resampled)}): {resampled}")
    return resampled

# Step 2 - RANDOM_API

The random api uses a prompt to draw initial candidates from the LLM API. In future, some additional subcategories should be enforced for better initital draws. 

In [13]:
# --- RANDOM API ---
def random_api(n):
    print(f"\n[RANDOM API] Generating {n} initial candidates...")
    prompt = (
        f"Generate {n} movie reviews: roughly 1/3 positive, 1/3 negative, 1/3 mixed sentiment. "
        "Each review should be 1-2 sentences. Around 10-15 words long"
        "Each review should relate to one or more of these categories: {', '.join(CATEGORIES)}. "
        "Make each generated review unique and diverse"
        "Return ONLY a numbered list, one review per line."
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    lines = [l.strip() for l in response.content[0].text.strip().split("\n") if l.strip()]
    candidates = []
    for line in lines:
        parts = line.split('. ', 1)
        candidates.append(parts[1].strip() if len(parts) == 2 and parts[0].isdigit() else line)
    candidates = candidates[:n]
    for i, c in enumerate(candidates, 1):
        print(f"  {i}. {c}")
    return candidates

# Step 3 - VARIATION API

The VARIATION_API is responsible for drawing a sample from the DP_NN_HISTOGRAM, and then using the LLM API to add some variation to the drawn set of samples. This just helps to add some additional variation to the samples.



In [14]:
# --- VARIATION API ---
# VARIATION_API now takes the full resampled pool
def variation_api(text, n):
    prompt = (
        f"Generate {n} diverse variations of the following movie review, "
        "Each review should relate to one or more of these categories: {', '.join(CATEGORIES)}. "
        "Each review should be 1-2 sentences. Around 10-15 words long"
        "each with the same general sentiment but different wording and style. "
        "Return ONLY a numbered list, one review per line.\n\n"
        f"Review: {text}"
    )
    # note sure if following helps?: 
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        #temperature=0.9,# Add some temperature to get more diverse variation
        messages=[{"role": "user", "content": prompt}]
    )
    lines = [l.strip() for l in response.content[0].text.strip().split("\n") if l.strip()]
    candidates = []
    for line in lines:
        parts = line.split('. ', 1)
        candidates.append(parts[1].strip() if len(parts) == 2 and parts[0].isdigit() else line)
    return candidates[:n]

# Step 4 - Run the Loop

Generates random reviews, finds the most similar records, adds variation to them, normally repeats for a certain number of runs. 

In [15]:
# Draw initial candidates
candidates = random_api(N_CANDIDATES)

for t in range(T):
    print(f"\n{'='*60}\nITERATION T={t+1}\n{'='*60}")
    resampled = dp_nn_histogram(candidates, private_data, EPSILON, n_resample=len(private_data))
    
    print(f"\n[VARIATION API]")
    candidates = []
    for text in resampled:
        varied = variation_api([text], n=N_VARIATIONS)
        candidates.extend(varied)
        print(f"  - {varied[0]}")

# Final synthetic dataset = last iteration's candidates
synthetic_data = candidates


[RANDOM API] Generating 1000 initial candidates...
  1. This action-packed thriller delivers heart-pounding sequences with stunning practical effects and compelling characters.
  2. The romantic comedy falls flat with predictable plot points and lackluster chemistry between leads.
  3. While the horror elements terrify, the weak storyline undermines an otherwise atmospheric experience.
  4. A masterful drama showcasing incredible performances and thought-provoking themes about human resilience.
  5. The sci-fi adventure disappoints with confusing world-building despite impressive visual effects throughout.
  6. Great comedy timing saves this film, though some jokes feel forced and outdated.
  7. This fantasy epic creates a breathtaking world filled with wonder and magical storytelling.
  8. The documentary presents compelling evidence but suffers from repetitive pacing and unclear focus.
  9. Brilliant cinematography elevates this thriller, but the convoluted plot leaves viewers confu

# Step 5 - Display results

Note the algorithm produces a distribution of synthetic data, but each one isn't matched to it's closest real 

In [18]:
# --- RESULTS TABLE ---
from scipy.optimize import linear_sum_assignment

def render_table(private_data, synthetic_data):
    real_embs = encoder.encode(private_data, normalize_embeddings=True)
    syn_embs  = encoder.encode(synthetic_data, normalize_embeddings=True)
    sim_matrix = real_embs @ syn_embs.T  # (n_real, n_syn)

    # Hungarian algorithm — convert similarity to cost (it minimises)
    cost_matrix = 1 - sim_matrix
    real_idx, syn_idx = linear_sum_assignment(cost_matrix)

    rows = []
    for i, j in zip(real_idx, syn_idx):
        best_sim = sim_matrix[i, j]
        conf = "High" if best_sim >= 0.8 else "Medium" if best_sim >= 0.6 else "Low"
        rows.append({
            "Real Review":     private_data[i],
            "Synthetic Match": synthetic_data[j],
            "Cosine Sim":      round(float(best_sim), 3),
            "Confidence":      conf
        })

    df = pd.DataFrame(rows)
    tbl = (
        GT(df)
        .tab_header(
            title="AUG-PE Results",
            subtitle=f"DP ε={EPSILON} — Budget spent: {EPSILON * T:.2f} ({T} iterations)"
        )
        .tab_style(style=style.fill(color="#d4edda"), locations=loc.body(rows=df.index[df["Confidence"] == "High"].tolist()))
        .tab_style(style=style.fill(color="#fff3cd"), locations=loc.body(rows=df.index[df["Confidence"] == "Medium"].tolist()))
        .tab_style(style=style.fill(color="#f8d7da"), locations=loc.body(rows=df.index[df["Confidence"] == "Low"].tolist()))
        .fmt_number(columns="Cosine Sim", decimals=3)
        .cols_width({"Real Review": "35%", "Synthetic Match": "35%", "Cosine Sim": "15%", "Confidence": "15%"})
    )
    return tbl

# At the end of the script:
render_table(private_data, candidates)

GT(_tbl_data=                                         Real Review                                    Synthetic Match  Cosine Sim  \
0  The lead actor delivered an absolutely mesmeri...  The actors' brilliant portrayals turn this mov...       0.792   
1  The storyline was predictable and full of plot...  Bad writing combined with uninspired direction...       0.683   
2  The cinematography was breathtaking, every fra...  Brilliant shot framing with outstanding visual...       0.747   
3  The CGI was cheap and unconvincing, completely...  Incredible visual artistry is undermined by th...       0.656   
4  The plot was gripping and kept me on the edge ...  Weak screenplay and lazy filmmaking ruin what ...       0.621   
5  Stunning visuals throughout but the story made...  Amazing CGI can't save this confusing mess of ...       0.781   
6  The cast was phenomenal and clearly committed ...  Spectacular production values fail to compensa...       0.629   
7  A visual masterpiece with a twisting plot and ...  Exceptional visual storytelling combined with ...       0.769   
8  Terrible acting, a nonsensical storyline, and ...  Gorgeous special effects are wasted on this in...       0.683   
9  The actors clearly tried their best but the we...  Impressive special effects failed to compensat...       0.701   

  Confidence  
0     Medium  
1     Medium  
2     Medium  
3     Medium  
4     Medium  
5     Medium  
6     Medium  
7     Medium  
8     Medium  
9     Medium  , _body=<great_tables._gt_data.Body object at 0x7e46f2053e90>, _boxhead=Boxhead([ColInfo(var='Real Review', type=<ColInfoTypeEnum.default: 1>, column_label='Real Review', column_align='left', column_width='35%'), ColInfo(var='Synthetic Match', type=<ColInfoTypeEnum.default: 1>, column_label='Synthetic Match', column_align='left', column_width='35%'), ColInfo(var='Cosine Sim', type=<ColInfoTypeEnum.default: 1>, column_label='Cosine Sim', column_align='right', column_width='15%'), ColInfo(var='Confidence', type=<ColInfoTypeEnum.default: 1>, column_label='Confidence', column_align='left', column_width='15%')]), _stub=<great_tables._gt_data.Stub object at 0x7e46f6036390>, _spanners=Spanners([]), _heading=Heading(title='AUG-PE Results', subtitle='DP ε=100000 — Budget spent: 500000.00 (5 iterations)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e46f2011f90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e46f2011a10>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=0, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=1, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=2, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=3, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=4, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=5, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=6, colnum=None, styles=[CellStyleFill(color='#fff3cd')]), StyleInfo(locname=LocBody(columns=None, rows=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], mask=None), grpname=None, colname='Real Review', rownum=7, 

# Improvements

- This was put together really quickly, need to refer to the [paper](https://alphapav.github.io/augpe-dpapitext/) or better use their code. There are defintely issues.
- Draw multiples Synethtic samples per real dataset to get a better distribution
- Add some tests on a benchmark like a sentiment analysis.
- I added a sentiment penalty for comparison which is a little hacky. Instead there must be some sentiment aware encodings?
- Do some fine tuning on the encoding model?
- Instead of voting for the nearest neighbour, spread the vote weighted across all neighbours? Should still be the same DP?

# Key Issue
- the distribution will never match back 1 to 1. This is because of the DP gaurantee. 
- Therefore, the best you can do is analyse the text data in aggregate, not link back to the records that spawn say the poor plot to understand more about the person who didn't like the plot. 
- The algorithm is super sensitive to the ability of the encoding model.